In [ ]:
import numpy as np
import pandas as pd
import nibabel as nb
import statsmodels.api as sm
import statsmodels.formula.api as smf
import statsmodels.regression.mixed_linear_model as MixedLM
import scipy.io 
from glob import glob
from os.path import join as pjoin
import os
import matplotlib.pyplot as plt
# setup folders and files to look for
base_dir = '/mnt/tambinidata/sleepstudy/'
analysis_dir = base_dir + 'analysis'
data_dir = pjoin(base_dir, 'data')
der_dir = pjoin(data_dir, 'derivatives')
Nses = [1,2,3,4]
ss_list = ['sub-102','sub-103', 'sub-105', 'sub-106', 'sub-107', 'sub-108', 'sub-111', 'sub-112', 'sub-113', 'sub-114',
'sub-115', 'sub-116', 'sub-118', 'sub-120', 'sub-121', 'sub-123', 'sub-124', 'sub-125', 'sub-126', 'sub-127','sub-130',
'sub-131','sub-132','sub-133','sub-135', 'sub-136', 'sub-137', 'sub-138', 'sub-141', 'sub-142', 'sub-144', 'sub-145',
'sub-148', 'sub-149','sub-152', 'sub-153']
Ns = 36

model_name = '/group/ospan_vs_math'
contrast = 'v*_am-pm_ospan_vs_math.nii.gz'
out_name = 'ospan_vs_math'

# load in other misc info - can remove
# dates = scipy.io.loadmat(pjoin(analysis_dir, 'ss_date_values.mat'))
# dates = dates['date_values']
info = scipy.io.loadmat(pjoin(analysis_dir, 'info_python.mat'))
info = info['info']
print(info)


In [ ]:
# modify to define group-level 
# you are using
# mask = '/mnt/tambinidata/sleepstudy/data/derivatives/group/ospan_vs_math/allsessions/allsub/positive/cluster_index_tfce_mask.nii.gz'
mask_template = glob(pjoin('/mnt/tambinidata/sleepstudy/data/derivatives/group', 'MNI_brainmask_ds.nii*'))
assert(len(mask_template)==1)

exclude = nb.load(mask_template[0]).get_data() > .5
# exclude = exclude.astype('int')
mask = exclude==1


In [ ]:
X_sub=[]
# cntr will instead be total # of datapoints you are loading in
cntr=0
Y = np.zeros((np.sum(mask), np.sum(info)), dtype=int )
# loop over subjects
visit=[]
print(np.arange(0,Ns))
for iss in np.arange(0,Ns):

    ss = ss_list[iss]
    print(ss)
    s_info = info[iss,:]
    ses_list = np.where(s_info>0)
    ses_name=s_info[ses_list]

    # loop over sessions
    for ises in np.where(s_info==1)[0]: #np.arange(0, Nses[iss]):
        print(ises)
        #print(ses_str)
        #print(run_list)
        run = 0
        ses_dir = der_dir + model_name
        # load in each contrast file
        contrast='/v' + str(ises+1) +'_' + ss + '_am-pm_ospan_vs_math.nii.gz'
        visit.append(ises)
        con_file = glob(ses_dir+ contrast)
        assert(len(con_file)==1)
        basename = os.path.basename(con_file[0])
        new_file = str(con_file[0])
        
        temp = nb.load(new_file).get_data()# this actually loads in data
        Y[:,cntr]= temp[mask] # just select voxels in mask
        X_sub.append(iss+1)
        cntr=cntr+1

In [ ]:
Nvox = Y.shape[0]
save_dir= '/mnt/tambinidata/sleepstudy/data/derivatives/group/ospan_vs_math/am-pm/visiteffect_wholebrain/'
# Output file names
tmap_name = pjoin(save_dir, 'Tstat_' + out_name + '_Time_mixedLM.nii.gz')
pmap_name = pjoin(save_dir, 'Pmap1m_' + out_name + '_Time_mixedLM.nii.gz')

print(Nvox)
Tvals = np.zeros((Nvox))
Pvals = np.ones((Nvox))
Tval_visit= np.zeros((Nvox))
Pval_visit= np.zeros((Nvox))
# X_date = np.array(X_date)
# X_mot = np.array(X_mot)
# X_sub = np.array(X_sub)
# X_run = np.array(X_run)
Y_name = 'Y'
X_name = 'visit'


import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning
warnings.simplefilter('ignore', ConvergenceWarning)
    
for ivox in np.arange(0, Nvox):

    # setup dataframe w/ variables for each voxels. In your case you'll have Y (overnight changes),
    # X-session (indicating first/second sessions), and X-sub (indicating subject ID) 
    d = {'Y': Y[ivox,:], 'visit': visit, 'IDS': X_sub} ## not sure what the info format should be
    df = pd.DataFrame(d)

    # setup mixed model and fit it
    md = MixedLM.MixedLM.from_formula('{} ~ {} '.format(Y_name,X_name), \
                                        groups = df['IDS'], data = df)


    if np.var(Y[ivox,:] ) == 0:
        print(ivox)
        continue
    
    mdf = md.fit() 
    mdf.summary()

    # populate these vectors w/ T, P values for each voxel
    Tvals[ivox] = mdf.tvalues.Intercept
    Pvals[ivox] = mdf.pvalues.Intercept
    Tval_visit[ivox] == mdf.tvalues.visit
    Pval_visit[ivox] == mdf.pvalues.visit
    if np.remainder(ivox, 100)==0:
        print(str(ivox) + '/' + str(Nvox) )

In [ ]:
Y[ivox,:]

In [ ]:
#mdf.summary()
mdf.tvalues

In [ ]:
out_name

In [ ]:
print(ivox)
aff = nb.load(mask_template[0]).affine

# put T, P values into 3d matrix instead of 2d
Tmap = np.zeros(mask.shape)
Pmap = np.zeros(mask.shape)

Tmap_visit = np.zeros(mask.shape)
Pmap_visit = np.zeros(mask.shape)

Tmap[mask] = Tvals
Pmap[mask] = 1-Pvals

Tmap_visit[mask] = Tval_visit
Pmap_visit[mask] = 1-Pval_visit
# save output files
tmap_name = pjoin(save_dir, 'Tstat_' + out_name + '_Time_mixedLM.nii.gz')
pmap_name = pjoin(save_dir, 'Pmap1m_' + out_name + '_Time_mixedLM.nii.gz')
tmap_visit_name = pjoin(save_dir, 'Tstat_' + out_name + '_Time_mixedLM_visit.nii.gz')
pmap_visit_name = pjoin(save_dir, 'Pmap1m_' + out_name + '_Time_mixedLM_visit.nii.gz')

In [ ]:

new_img = nb.Nifti1Image(Tmap, aff)
new_img.to_filename(tmap_name)

new_img = nb.Nifti1Image(Pmap, aff)
new_img.to_filename(pmap_name)


new_img = nb.Nifti1Image(Tmap_visit, aff)
new_img.to_filename(tmap_visit_name)

new_img = nb.Nifti1Image(Pmap_visit, aff)
new_img.to_filename(pmap_visit_name)
#np.sum(exclude>0)
#X_sub
#exclude.astype('int')

#np.where(np.sum(info[:,:,1],axis=1)>0)
#np.where(info[7,:,1])
#info[:,:,0]